# Initial model — GPU experiment

This notebook runs the initial entity-resolution model with the CUDA-enabled pipeline in `src/run_pipeline.py`. The source module performs a hard CUDA check, uses CuPyX for sparse TF-IDF similarity, and trains XGBoost with `device="cuda"`.

The expensive training and prediction cells are intentionally not executed automatically.

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from either initial_model/ or Amazon_ml/.
project_dir = Path.cwd()
if not (project_dir / 'src' / 'run_pipeline.py').exists():
    project_dir = project_dir / 'initial_model'
if not (project_dir / 'src' / 'run_pipeline.py').exists():
    raise FileNotFoundError('Run this notebook from Amazon_ml/ or Amazon_ml/initial_model/.')

sys.path.insert(0, str(project_dir))
from src import run_pipeline as pipeline

SAMPLE_SIZE = 20_000
print(f'Project: {project_dir}')

## Verify the CUDA runtime

This cell should be run before loading data. It fails immediately if CuPy cannot see an NVIDIA GPU.

In [ ]:
if not pipeline.cp.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Install initial_model/requirements.txt and check the NVIDIA driver.')
device = pipeline.cp.cuda.Device()
print(f'CUDA device: {pipeline.cuda_device_name(device)}')
print(f'CuPy version: {pipeline.cp.__version__}')
print(f'XGBoost device setting: cuda')

## Train and validate

`run_train` samples Source 1, builds the GPU TF-IDF blocker, trains XGBoost on CUDA, optimises the F₀.₅ threshold, evaluates the validation split, and saves the model checkpoint.

In [ ]:
model, threshold = pipeline.run_train(sample_size=SAMPLE_SIZE)
print(f'Validation threshold: {threshold:.6f}')
print(f'Model checkpoint: {pipeline.MODEL_PATH}')

## Generate test predictions

The test set is processed country-by-country. Existing country checkpoints are reused, so this cell can safely be rerun after an interrupted job.

In [ ]:
matches, candidates = pipeline._predict_test_by_country(model, threshold)
print(f'S1 predictions: {len(matches):,}')
print(f'Candidate rows: {len(candidates):,}')
print(f'Output directory: {pipeline.OUTPUT_DIR}')

## Inspect the generated results

In [ ]:
import pandas as pd

matching_path = Path(pipeline.OUTPUT_DIR) / 'matching_results.tsv'
candidate_path = Path(pipeline.OUTPUT_DIR) / 'candidate_pairs.tsv'
matching_preview = pd.read_csv(matching_path, sep='\t', nrows=10)
candidate_preview = pd.read_csv(candidate_path, sep='\t', nrows=10)
display(matching_preview)
display(candidate_preview)

### Predict-only workflow

After a model checkpoint exists, use the following instead of retraining:

```python
pipeline.run_predict()
```